# IOAI — 2025 Summer National Catacomb Adventure (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CreepyCatacombs Gymnasium 패키지는 노트북 첫 셀이 자동 설치합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Catacomb Adventure (Kripta kaland) — RL

> **HAIO 2025 — Summer National Finals (RL)**

Escape the **CreepyCatacombs** maze (Gymnasium). Submit a **Q-table** (`submission.npy`, shape `[n_states, n_actions]`); grading runs its **greedy policy** on the canonical `seed=2025` maze and reports the **mean episode reward over 5 runs** (higher is better; each step costs −1, so the optimal escape is **−8**).

⚠️ **The baseline below is an *undertrained* Monte-Carlo agent (~50 episodes): it does escape, but by a wandering, suboptimal route (reward ≈ −20).** Improve it — **train longer**, or use **SARSA / Q-learning** — to approach the optimal **−8**. See the model solution.

(Reward higher = better; −8 is optimal, −300 means it never escaped.)

In [ ]:
import os, logging, warnings
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')   # 헤드리스: ALSA 오디오 오류 억제
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')   # rgb_array 오프스크린 렌더는 그대로
logging.disable(logging.INFO)                       # pygame 렌더러 로그 스팸 억제
warnings.filterwarnings('ignore', message='.*pkg_resources is deprecated.*')
!pip install -qq creepy-catacombs-s1==0.1.4
import numpy as np, creepy_catacombs_s1, gymnasium as gym

In [ ]:
env = gym.make('CreepyCatacombs-v0')
env.reset(seed=2025)
nS, nA = env.observation_space.n, env.action_space.n

def monte_carlo_control(episodes, gamma=0.99, seed=0):
    rng = np.random.RandomState(seed)
    Q = np.zeros((nS, nA)); C = np.zeros((nS, nA))
    for ep in range(episodes):
        eps = max(0.1, 1.0 - ep/episodes)              # explore -> exploit
        o, _ = env.reset(seed=2025); traj = []
        for _ in range(300):
            a = rng.randint(nA) if rng.rand() < eps else int(np.argmax(Q[o]))
            o2, r, d, t, _ = env.step(a); traj.append((o, a, r)); o = o2
            if d or t: break
        G = 0
        for s, a, r in reversed(traj):                 # first-visit-ish MC update
            G = gamma*G + r; C[s, a] += 1; Q[s, a] += (G - Q[s, a]) / C[s, a]
    return Q

# BASELINE: only ~50 episodes -> undertrained (suboptimal path)
Q = monte_carlo_control(episodes=50)

In [ ]:
def mean_reward(Q, seed=2025, episodes=5):
    tot = 0.0
    for _ in range(episodes):
        o, _ = env.reset(seed=seed)
        for _ in range(300):
            o, r, d, t, _ = env.step(int(np.argmax(Q[o]))); tot += r
            if d or t: break
    return tot/episodes
np.save('submission.npy', Q)
print('wrote submission.npy', Q.shape, '| seed-2025 mean reward:', mean_reward(Q))
print('(undertrained ~-20 — train more / use SARSA to reach the optimal -8)')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)